# Field Development API Mastery


In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_project_root():
    env_root = os.environ.get("NEQSIM_PROJECT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).resolve())
    try:
        candidates.append(Path(__vsc_ipynb_file__).resolve())
    except NameError:
        candidates.append(Path.cwd().resolve())
    expanded = []
    for candidate in candidates:
        expanded.extend([candidate] + list(candidate.parents))
    for candidate in expanded:
        if (candidate / "pom.xml").exists() and (candidate / "devtools" / "neqsim_dev_setup.py").exists():
            return candidate
    raise RuntimeError("Could not find NeqSim project root")


try:
    NOTEBOOK_DIR = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd().resolve()

FIGURES_DIR = NOTEBOOK_DIR.parent / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "devtools"))

NEQSIM_AVAILABLE = False
NEQSIM_ERROR = ""
try:
    from neqsim_dev_setup import neqsim_init

    ns = neqsim_init(project_root=PROJECT_ROOT, recompile=False, verbose=False)
    JClass = ns.JClass
    NEQSIM_AVAILABLE = True
except Exception as exc:
    ns = None
    JClass = None
    NEQSIM_ERROR = str(exc)

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Figures directory: {FIGURES_DIR}")
print(f"NeqSim Java bridge available: {NEQSIM_AVAILABLE}")
if NEQSIM_ERROR:
    print(f"NeqSim bridge warning: {NEQSIM_ERROR}")


Notebook directory: C:\Users\ESOL\Documents\GitHub\neqsim\neqsim-paperlab\books\tpg4230_field_development_and_operations_2026\chapters\ch24_computational_tools_neqsim\notebooks
Figures directory: C:\Users\ESOL\Documents\GitHub\neqsim\neqsim-paperlab\books\tpg4230_field_development_and_operations_2026\chapters\ch24_computational_tools_neqsim\figures
NeqSim Java bridge available: True


## API Availability Check


In [2]:
class_names = [
    "neqsim.process.fielddevelopment.concept.FieldConcept",
    "neqsim.process.fielddevelopment.concept.ReservoirInput",
    "neqsim.process.fielddevelopment.concept.WellsInput",
    "neqsim.process.fielddevelopment.concept.InfrastructureInput",
    "neqsim.process.fielddevelopment.evaluation.ConceptEvaluator",
    "neqsim.process.fielddevelopment.evaluation.DevelopmentOptionRanker",
    "neqsim.process.fielddevelopment.screening.FlowAssuranceScreener",
    "neqsim.process.fielddevelopment.screening.GasLiftCalculator",
    "neqsim.process.fielddevelopment.network.NetworkSolver",
    "neqsim.process.fielddevelopment.tieback.TiebackAnalyzer",
    "neqsim.process.fielddevelopment.subsea.SubseaProductionSystem",
    "neqsim.process.fielddevelopment.economics.CashFlowEngine",
]
available = []
for name in class_names:
    ok = False
    if JClass is not None:
        try:
            JClass(name)
            ok = True
        except Exception:
            ok = False
    available.append((name.split(".")[-1], ok))
for cls, ok in available:
    status = "available" if ok else "not loaded"
    print(f"{cls:<28} {status}")


FieldConcept                 available
ReservoirInput               available
WellsInput                   available
InfrastructureInput          available
ConceptEvaluator             available
DevelopmentOptionRanker      available
FlowAssuranceScreener        available
GasLiftCalculator            available
NetworkSolver                available
TiebackAnalyzer              available
SubseaProductionSystem       available
CashFlowEngine               available


## Capability Coverage


In [3]:
packages = ["concept", "screening", "evaluation", "economics", "facility", "network", "subsea", "tieback", "workflow"]
counts = np.array([4, 10, 9, 8, 5, 3, 2, 4, 2])
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(packages, counts, color="#577590")
ax.set_ylabel("Class count in package")
ax.set_title("NeqSim Field-Development API Coverage by Package")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.3)
fig.savefig(FIGURES_DIR / "ch24_fielddev_api_package_coverage.png", dpi=150, bbox_inches="tight")
plt.close(fig)


**Discussion (Figure ch24_fielddev_api_package_coverage.png).** *Observation.* The API spans concept definition, screening, evaluation, economics, facilities, network, subsea, tieback and workflow classes. *Mechanism.* The field-development module separates data objects, calculators and orchestration workflows into packages. *Implication.* Users can start with simple screening and progressively connect more detailed process models. *Recommendation.* Teach the package map before asking students to build a full concept workflow.


## API Workflow Map


In [4]:
steps = ["Inputs", "Screen", "Simulate", "Evaluate", "Rank", "Report"]
x = np.arange(len(steps))
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(x, np.zeros_like(x), color="#264653", linewidth=2)
ax.scatter(x, np.zeros_like(x), s=900, color="#a8dadc", edgecolors="#1d3557", linewidths=2)
for xi, step in zip(x, steps):
    ax.text(xi, 0, step, ha="center", va="center", fontweight="bold")
ax.set_xlim(-0.6, len(steps) - 0.4)
ax.set_ylim(-0.7, 0.7)
ax.axis("off")
ax.set_title("Field-Development API Workflow")
fig.savefig(FIGURES_DIR / "ch24_fielddev_api_workflow.png", dpi=150, bbox_inches="tight")
plt.close(fig)


**Discussion (Figure ch24_fielddev_api_workflow.png).** *Observation.* The workflow runs from structured inputs through screening, simulation, evaluation, ranking and reporting. *Mechanism.* Each stage consumes the previous stage's outputs and adds more physical or economic detail. *Implication.* Notebook examples should expose intermediate objects rather than only final NPV values. *Recommendation.* When debugging, validate each stage independently before running the full workflow.


## Exercises

1. Identify which package owns concept input data and which package owns ranking.
2. Explain why API availability checks should catch exceptions rather than failing immediately.
3. Add a hypothetical RiskRegister class to the package map and choose its package.
4. Describe when to use direct JClass access instead of a preloaded Python alias.
5. Write a six-stage pseudocode workflow for concept selection.
